# 🦜 Personal Resource Assistant — RAG with LangChain, OpenAI & ChromaDB — Explained

**What this notebook does:** the exact same RAG (Retrieval-Augmented Generation) assistant as `RAG_Personal_Resource_Assistant_Langchain,_Cohere_and_ChromaDB.ipynb`, rewritten to use the **OpenAI** ecosystem instead of Cohere — `OpenAIEmbeddings` for retrieval and `ChatOpenAI` for generation, both from `langchain-openai`. Everything else — PDF extraction, chunking, ChromaDB, the LCEL pipeline, and `.invoke()` / `.batch()` / `.stream()` — stays conceptually identical. Each code cell below has a short explanation directly above it.

## Installing necessary libraries

OpenAI does not offer a free trial tier the way Cohere does — you'll need a funded API key. Generate one from [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

**Where to put the key.** This notebook reads it from two places, so the same file runs in either environment:

- **Colab** — add it under 🔑 *Secrets* in the left sidebar, named `OPENAI_API_KEY`, with notebook access enabled.
- **Locally** (Jupyter / VS Code / `nbconvert`) — export it in the shell before launching: `export OPENAI_API_KEY=sk-...` (PowerShell: `$env:OPENAI_API_KEY = "sk-..."`).

Never paste the key into a cell — it would get committed along with the notebook.

### 📦 Cell — Install the libraries

Same as the Cohere version, but `langchain-cohere` is swapped for **`langchain-openai`** — LangChain's integration package for OpenAI's chat and embedding models. Everything else (`pdfminer.six` for PDFs, `chromadb` for the vector store, the text splitter) is unchanged, since none of that is provider-specific.

One extra change from the Cohere notebook: Chroma now comes from its own **`langchain-chroma`** package instead of `langchain-community`. The `langchain_community.vectorstores.Chroma` class is deprecated and was written against chromadb 0.4.x, so it breaks against current chromadb releases — `langchain-chroma` is the maintained replacement with the same API.

In [1]:
!pip install langchain-openai langchain langchain-chroma pdfminer.six chromadb langchain-text-splitters


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


langchain-openai: Enables integration of OpenAI's language and embedding models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

langchain-chroma: The maintained LangChain wrapper around ChromaDB — gives the vector store its `from_documents` / `as_retriever` / `similarity_search` interface.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

### 🧰 Cell — Import everything, and set the OpenAI API key

Reads the OpenAI API key and sets it as an environment variable — `ChatOpenAI` and `OpenAIEmbeddings` both pick this up automatically, the same pattern `ChatCohere` and `CohereEmbeddings` used for `COHERE_API_KEY`.

The `try` / `except ImportError` is what makes this notebook portable: `google.colab` only exists inside Colab, so in Colab the key comes from Colab's secret storage, and anywhere else the import fails harmlessly and it falls back to whatever `OPENAI_API_KEY` is already in the environment. The `assert` then fails fast with a readable message instead of letting the first API call blow up with a confusing auth error.

Everything else is the same import list as before, just with `langchain_openai` swapped in for `langchain_cohere`, and `Chroma` imported from `langchain_chroma`.

In [2]:
import os

# Portable key loading: Colab secret when running in Colab, otherwise whatever
# OPENAI_API_KEY is already exported in the shell.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

# Fail immediately with a clear message rather than on the first API call
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set - add it to Colab Secrets or export it in your shell."
)

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

### 📂 Cell — Decide where files live, and fetch the two papers

The Cohere notebook hard-coded Colab paths like `/content/1706.03762v7.pdf`, which quietly assumed you had already uploaded the PDFs by hand. This cell replaces that with two portable pieces:

- **`BASE_DIR`** — the notebook's working directory. In Colab that *is* `/content`, so nothing changes there; run it locally and everything lands next to the notebook instead.
- **auto-download** — the two papers are pulled straight from arXiv if they aren't on disk yet, so the notebook is self-contained and needs no manual upload step.

The papers are **"Attention Is All You Need"** (the transformer paper) and **"You Only Look Once"** (YOLO, real-time object detection) — two clearly different topics, which makes it obvious later on whether retrieval is actually picking the right document for a question.

In [3]:
from pathlib import Path
import urllib.request

# The notebook's working directory: "/content" in Colab, the notebook folder locally
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The two papers this assistant answers questions about
PAPERS = {
    "1706.03762v7.pdf": "https://arxiv.org/pdf/1706.03762v7",  # Attention Is All You Need
    "1506.02640v5.pdf": "https://arxiv.org/pdf/1506.02640v5",  # YOLO
}

# Download anything missing, and collect the local paths for the next cells
pdf_paths = []
for name, url in PAPERS.items():
    path = DATA_DIR / name
    if not path.exists():
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(url, path)
    pdf_paths.append(str(path))
    print(f"{name}  ({path.stat().st_size / 1e6:.1f} MB)")

1706.03762v7.pdf  (2.2 MB)
1506.02640v5.pdf  (5.3 MB)


## VectorDB setup

### 🗄️ Cell — Set up where the vector database lives, and which embedding model to use

`persist_directory` works exactly the same as the Cohere version — the folder on disk where Chroma saves its data, so it survives a restart. The only change is that it now hangs off `BASE_DIR` instead of a hard-coded `/content/...` path. `OpenAIEmbeddings` replaces `CohereEmbeddings`, using `text-embedding-3-small` — OpenAI's current, cost-efficient embedding model, which produces 1536-dimensional vectors. As always, this exact same model has to embed both the PDF chunks *and* every question later, or the comparisons wouldn't mean anything.

In [4]:
# Define the directory where the Chroma database will persist data
# Built from BASE_DIR so this works in Colab and locally alike
persist_directory = str(BASE_DIR / "chroma_db_openai")

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

We are processing 2 research papers on transformers and yolo. You can use your own PDFs — just point `PAPERS` above at them, or drop them into `data/` and extend `pdf_paths`.

### 📄 Cell — Turn each PDF into chunks, and store them in the vector database

Conceptually identical to the Cohere version, and deliberately so — chunking and storage have nothing to do with which model provider you use. Each PDF's text is extracted, cleaned, split into 2048-character chunks with 512 characters of overlap, wrapped in `Document` objects tagged with their source file, then embedded (this time by OpenAI) and saved into the persistent Chroma database.

Two mechanical fixes over the original loop, both about what happens when you **re-run the cell**:

- The original called `Chroma.from_documents(...)` *inside* the loop, once per PDF, resetting `docs = []` each time. That happens to work — the second call appends to the same persisted collection — but collecting every chunk first and writing once is clearer and makes one round of embedding calls instead of two.
- Writing into a Chroma directory that already holds data **adds** to it, so running this cell twice would leave two copies of every chunk and skew retrieval. Emptying the collection first makes the cell idempotent: run it as many times as you like and the database always ends up with exactly one copy. `reset_collection()` does this through Chroma's own API rather than deleting the folder — on Windows a `shutil.rmtree` would hit a `PermissionError`, because a Chroma client created earlier in the session still has the index files open.

In [21]:
# Writing into an existing Chroma directory appends, so a re-run would duplicate
# every chunk. Emptying the collection first keeps this cell idempotent.
# (reset_collection goes through Chroma's own API, so it works even while a
# client still holds the files open - deleting the folder on Windows would not.)
Chroma(persist_directory=persist_directory, embedding_function=embedding).reset_collection()

# Initialize a list to store document chunks from every PDF
docs = []

# Loop through the list of PDF files to process
for pdf_name in pdf_paths:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

    # Clean the extracted text by removing newline characters and joining into a single string
    cleaned_text = " ".join(text.split("\n"))

    # Create a text splitter to divide the text into manageable chunks
    # Each chunk has a maximum size of 2048 characters with a 512-character overlap
    splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

    # Split the cleaned text into chunks and wrap each chunk in a Document object
    chunks = splitter.split_text(cleaned_text)
    for chunk in chunks:
        docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    print(f"{Path(pdf_name).name}: {len(text):,} chars -> {len(chunks)} chunks")

# Create a Chroma collection from the processed documents
# Use the specified persist directory and embedding model for storage and retrieval
vector_collection_fixed_size = Chroma.from_documents(
    documents=docs,
    persist_directory=persist_directory,
    embedding=embedding
)

print(f"\nStored {len(docs)} chunks in {persist_directory}")

1706.03762v7.pdf: 39,675 chars -> 26 chunks
1506.02640v5.pdf: 42,648 chars -> 28 chunks

Stored 54 chunks in C:\Users\Avado\Desktop\LearningGenAI\LangChain\chroma_db_openai


### 🔌 Cell — Reconnect to the vector database

Opens a handle to the same persistent Chroma database just filled in the previous cell — same `persist_directory`, same OpenAI embedding model.

In [22]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

### 🔍 Cell — Try a similarity search directly

Same sanity check as before: search the vector database directly for the single closest chunk (`k=1`) to “What is YOLO?”, with a relevance score — dense vector retrieval on its own, no LLM involved yet.

In [23]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=3)

[(Document(id='b85aae92-9ce3-4905-bbb3-5d80534c4c4b', metadata={'source': 'C:\\Users\\Avado\\Desktop\\LearningGenAI\\LangChain\\data\\1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results.

## How the embeddings are actually stored

Everything so far treated the vector database as a black box: text went in, relevant text came back out. This section opens it up. Nothing here is needed for the RAG pipeline to work — it exists to make "the embeddings are stored in Chroma" concrete, because that phrase hides three separate things:

1. **The vector itself** — a fixed-length list of floats, which is all the model's understanding of a chunk gets compressed into.
2. **The records** — the chunk text, its metadata, and its id, sitting in a plain SQLite database you can open with any SQLite client.
3. **The index** — the HNSW graph that makes search fast, which is a *derived* structure and, as we'll see, isn't even written to disk yet at this small scale.

Read them in that order and "vector store" stops being a black box.

### 🔬 Cell — Look at one stored embedding

`vectordb.get(include=[...])` pulls records straight out of the store without any similarity search — useful precisely because it shows what's *there* rather than what matches a query.

The important number is the shape: **54 chunks × 1536 floats**. Every chunk, whether it was 300 characters or 2048, becomes exactly 1536 numbers — that fixed width is what makes vectors comparable at all. Note also that the vectors come back **L2-normalised** (length 1.0); OpenAI returns them that way, which is why a dot product between two of them *is* their cosine similarity.

In [24]:
import numpy as np

# Pull raw records out of the store - no similarity search involved.
# include= is needed for embeddings: Chroma omits the vectors by default
# because they are by far the bulkiest part of each record.
raw = vectordb.get(include=["embeddings", "documents", "metadatas"])

E = np.array(raw["embeddings"])
print(f"chunks stored     : {len(raw['ids'])}")
print(f"embeddings shape  : {E.shape}   <- (n_chunks, dimensions)")
print(f"one vector is     : {E.shape[1]} numbers, whatever the chunk length")
print()

# Look at a single record end to end
i = 0
print(f"id                : {raw['ids'][i]}")
print(f"metadata          : {raw['metadatas'][i]}")
print(f"text (first 120)  : {raw['documents'][i][:120]!r}")
print(f"vector (first 8)  : {np.round(E[i][:8], 6).tolist()}")
print(f"vector min / max  : {E[i].min():.5f} / {E[i].max():.5f}")
print(f"vector L2 norm    : {np.linalg.norm(E[i]):.6f}  <- OpenAI returns unit vectors")
print()

# Because the vectors are unit length, a dot product IS cosine similarity.
# Chunks from the same paper should score higher against each other than
# chunks from different papers.
sources = [m["source"] for m in raw["metadatas"]]
paper = [s.split("/")[-1].split("\\")[-1] for s in sources]
sim = E @ E.T
same = [sim[a, b] for a in range(len(E)) for b in range(a + 1, len(E)) if paper[a] == paper[b]]
diff = [sim[a, b] for a in range(len(E)) for b in range(a + 1, len(E)) if paper[a] != paper[b]]
print(f"mean cosine, same paper      : {np.mean(same):.4f}")
print(f"mean cosine, different papers: {np.mean(diff):.4f}   <- lower, as it should be")
print()

print(f"storage cost      : {E.size:,} floats = {E.size * 4 / 1e6:.2f} MB as float32")

chunks stored     : 54
embeddings shape  : (54, 1536)   <- (n_chunks, dimensions)
one vector is     : 1536 numbers, whatever the chunk length

id                : cb3fe564-722f-4b2a-a231-16a9aed132a4
metadata          : {'source': 'C:\\Users\\Avado\\Desktop\\LearningGenAI\\LangChain\\data\\1706.03762v7.pdf'}
text (first 120)  : '3 2 0 2  g u A 2  ] L C . s c [  7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a  Provided proper attribution is provided, Google he'
vector (first 8)  : [0.001336, 0.018143, 0.012878, -0.004658, 0.010811, -0.054413, 0.012711, 0.038147]
vector min / max  : -0.08600 / 0.07617
vector L2 norm    : 1.000405  <- OpenAI returns unit vectors

mean cosine, same paper      : 0.6403
mean cosine, different papers: 0.3512   <- lower, as it should be

storage cost      : 82,944 floats = 0.33 MB as float32


### 🗃️ Cell — The files Chroma writes, and the tables inside them

A persistent Chroma directory is just **one SQLite database plus one folder per vector segment**. Worth noticing in the output:

- **`chroma.sqlite3`** holds the records. `collections` stores the dimension (1536) and the index config; `embedding_metadata` holds both your own metadata (`source`) and the chunk text under the reserved key `chroma:document` — two rows per chunk, hence 108 rows for 54 chunks.
- **The `embeddings` table has no vector column.** That surprises people. It maps ids to segments; the floats live elsewhere, which the next cell chases down.
- **You may see more than one segment folder than there are live segments.** `reset_collection()` in the chunking cell creates a *new* collection with a new id, and the previous collection's folder is left behind as garbage. The `live?` column marks which folder the current collection actually uses.

In [25]:
import sqlite3
from pathlib import Path

db_path = Path(persist_directory) / "chroma.sqlite3"
con = sqlite3.connect(db_path)

# --- which segment folders belong to the live collection? ---
live_segments = {row[0] for row in con.execute("SELECT id FROM segments")}

print("=== FILES ON DISK ===")
for f in sorted(Path(persist_directory).rglob("*")):
    if f.is_file():
        rel = f.relative_to(persist_directory)
        owner = rel.parts[0]
        flag = "" if owner.endswith(".sqlite3") else ("  live" if owner in live_segments else "  ORPHAN")
        print(f"{f.stat().st_size:>10,}  {rel}{flag}")

print("\n=== TABLES IN chroma.sqlite3 (non-empty) ===")
for (name,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    n = con.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    if n:
        print(f"{name:<34} {n:>5} rows")

print("\n=== collections ===")
cid, cname, dim = con.execute("SELECT id, name, dimension FROM collections").fetchone()
print(f"name={cname}  dimension={dim}  id={cid}")

print("\n=== segments (one per storage role) ===")
for sid, stype, scope in con.execute("SELECT id, type, scope FROM segments"):
    print(f"{scope:<9} {stype.split('/')[-1]:<28} {sid}")

print("\n=== embeddings table: note there is NO vector column ===")
print([d[0] for d in con.execute("SELECT * FROM embeddings LIMIT 1").description])

print("\n=== embedding_metadata: your metadata AND the chunk text ===")
for row in con.execute("SELECT id, key, substr(string_value,1,60) FROM embedding_metadata WHERE id=1"):
    print(row)

=== FILES ON DISK ===
   628,400  02b1cac8-8a1a-4e1b-be94-99120e55fb5b\data_level0.bin  live
       100  02b1cac8-8a1a-4e1b-be94-99120e55fb5b\header.bin  live
       400  02b1cac8-8a1a-4e1b-be94-99120e55fb5b\length.bin  live
         0  02b1cac8-8a1a-4e1b-be94-99120e55fb5b\link_lists.bin  live
   628,400  47d12da7-a11e-448d-950f-b29006ee42b9\data_level0.bin  ORPHAN
       100  47d12da7-a11e-448d-950f-b29006ee42b9\header.bin  ORPHAN
       400  47d12da7-a11e-448d-950f-b29006ee42b9\length.bin  ORPHAN
         0  47d12da7-a11e-448d-950f-b29006ee42b9\link_lists.bin  ORPHAN
   628,400  54aea318-072b-4e04-8e68-759c5a92e3ba\data_level0.bin  ORPHAN
       100  54aea318-072b-4e04-8e68-759c5a92e3ba\header.bin  ORPHAN
       400  54aea318-072b-4e04-8e68-759c5a92e3ba\length.bin  ORPHAN
         0  54aea318-072b-4e04-8e68-759c5a92e3ba\link_lists.bin  ORPHAN
   628,400  b7c7c7f4-f931-43e0-8cd9-557ef9564cdb\data_level0.bin  ORPHAN
       100  b7c7c7f4-f931-43e0-8cd9-557ef9564cdb\header.bin  ORPHAN
  

### 🧭 Cell — Where the floats really live: raw bytes, and an index that is still empty

This is the part that's genuinely surprising, and it's why the previous cell found no vector column.

**The vectors of record are BLOBs in `embeddings_queue`** — a write-ahead log. Each one is exactly **6144 bytes = 1536 × 4**, raw little-endian float32, with `encoding` literally spelling out `FLOAT32`. No compression, no quantisation. The cell decodes one with `struct.unpack` and checks it byte-for-byte against what the API returned a moment ago — they match exactly, which is the proof that this really is where the data sits.

**The HNSW index, meanwhile, is empty.** `header.bin` is a 100-byte hnswlib header, and parsing it shows `cur_element_count = 0`, `max_level = -1`, `entrypoint = -1` — an index with nothing in it. Yet `data_level0.bin` is 628,400 bytes of *pre-allocated* space, and the header explains that number exactly:

```
      132 bytes  neighbour links (4-byte count + 32 x 4-byte ids, since max_neighbors=16 -> maxM0=32)
+   6,144 bytes  the vector itself (1536 x float32)
+       8 bytes  the label (row id)
= ---------------
    6,284 bytes  per element  x  100 pre-allocated slots  =  628,400 bytes
```

So HNSW stores its **own second copy** of every vector, inline in the graph, so a search never has to jump back to SQLite. The reason it's still all zeros here is `sync_threshold = 1000` in the collection config: Chroma only flushes the graph to disk after 1000 pending writes. We wrote 54. Until then the file is a placeholder and the index is rebuilt in memory by replaying the log at load time — which is exactly why search worked earlier despite the index file being empty.

**The takeaway for interviews:** durability comes from the log, speed comes from the index, and the index is a disposable cache that can always be rebuilt from the log. That split is how most vector stores work, not a Chroma quirk.

In [26]:
import struct

# ---------- 1. The vectors of record: raw float32 BLOBs in the write-ahead log ----------
row = con.execute(
    "SELECT id, length(vector), encoding FROM embeddings_queue ORDER BY seq_id LIMIT 1"
).fetchone()
rec_id, nbytes, encoding = row
print("=== embeddings_queue (the write-ahead log) ===")
print(f"id            : {rec_id}")
print(f"encoding      : {encoding}")
print(f"blob size     : {nbytes:,} bytes = {nbytes // 4} floats x 4 bytes")
print()

blob, = con.execute("SELECT vector FROM embeddings_queue WHERE id = ?", (rec_id,)).fetchone()
decoded = np.array(struct.unpack(f"<{nbytes // 4}f", blob))   # '<' = little-endian, 'f' = float32
print(f"decoded first 5: {np.round(decoded[:5], 6).tolist()}")

# Prove this really is the source of truth: compare against the API result above
api_vector = E[raw["ids"].index(rec_id)]
print(f"identical to what vectordb.get() returned: {np.array_equal(decoded, api_vector)}")
print()

# ---------- 2. The HNSW index: parse its 100-byte header ----------
vector_seg = con.execute(
    "SELECT id FROM segments WHERE scope='VECTOR'"
).fetchone()[0]
seg_dir = Path(persist_directory) / vector_seg

header = (seg_dir / "header.bin").read_bytes()
# 4-byte Chroma format prefix, then the standard hnswlib header
fields = struct.unpack("<I6q2i3qdq", header)
(_prefix, offset_level0, max_elements, cur_element_count, size_per_element,
 label_offset, offset_data, max_level, entrypoint, maxM, maxM0, M,
 mult, ef_construction) = fields

print("=== HNSW header.bin ===")
print(f"max_elements      : {max_elements}    <- pre-allocated capacity")
print(f"cur_element_count : {cur_element_count}    <- how many are actually indexed on disk")
print(f"max_level         : {max_level}")
print(f"entrypoint        : {entrypoint}   (-1 means the graph is empty)")
print(f"M / maxM0         : {M} / {maxM0}")
print(f"ef_construction   : {ef_construction}")
print(f"size_per_element  : {size_per_element} bytes")
print(f"offset_data       : {offset_data}   label_offset: {label_offset}")
print()

# ---------- 3. Show that size_per_element is just links + vector + label ----------
links = offset_data                      # 4-byte neighbour count + maxM0 x 4-byte ids
vector_bytes = label_offset - offset_data  # the vector, inline in the graph
label_bytes = size_per_element - label_offset
data_file = seg_dir / "data_level0.bin"
print("=== data_level0.bin layout ===")
print(f"{links:>7,} bytes  neighbour links  (4 + {maxM0} x 4)")
print(f"{vector_bytes:>7,} bytes  vector inline    ({vector_bytes // 4} x float32)")
print(f"{label_bytes:>7,} bytes  label")
print(f"{size_per_element:>7,} bytes  per element")
print(f"{size_per_element:,} x {max_elements} slots = {size_per_element * max_elements:,} bytes"
      f"  |  actual file: {data_file.stat().st_size:,} bytes")
print()

print(f"So HNSW keeps a SECOND copy of every vector inline in the graph.")
print(f"It is all zeros right now because sync_threshold=1000 and we only wrote "
      f"{len(raw['ids'])} chunks -")
print(f"the index is rebuilt in memory from the log until that threshold is crossed.")

con.close()

=== embeddings_queue (the write-ahead log) ===
id            : cb3fe564-722f-4b2a-a231-16a9aed132a4
encoding      : FLOAT32
blob size     : 6,144 bytes = 1536 floats x 4 bytes

decoded first 5: [0.001336, 0.018143, 0.012878, -0.004658, 0.010811]
identical to what vectordb.get() returned: True

=== HNSW header.bin ===
max_elements      : 100    <- pre-allocated capacity
cur_element_count : 0    <- how many are actually indexed on disk
max_level         : -1
entrypoint        : -1   (-1 means the graph is empty)
M / maxM0         : 16 / 32
ef_construction   : 100
size_per_element  : 6284 bytes
offset_data       : 132   label_offset: 6276

=== data_level0.bin layout ===
    132 bytes  neighbour links  (4 + 32 x 4)
  6,144 bytes  vector inline    (1536 x float32)
      8 bytes  label
  6,284 bytes  per element
6,284 x 100 slots = 628,400 bytes  |  actual file: 628,400 bytes

So HNSW keeps a SECOND copy of every vector inline in the graph.
It is all zeros right now because sync_threshold=10

## RAG pipeline

### 🔗 Cell — Build the full RAG pipeline with LCEL

The same pipeline shape as the Cohere version, with the model swapped:

- **`llm`** — `ChatOpenAI` using `gpt-4o-mini`, with `temperature=0` for consistent, deterministic answers.
- **`prompt`** — the identical two-placeholder template (`{context}`, `{question}`), instructing the model to answer strictly from the given context.
- **`retrieval`** — unchanged: `RunnableParallel` runs `vectordb.as_retriever()` (fetch relevant chunks) and `RunnablePassthrough()` (keep the question as-is) side by side, producing `{"context": ..., "question": ...}`.
- **`chain = retrieval | prompt | llm | output_parser`** — the exact same LCEL pipe shape as the Cohere notebook; only the model inside it changed.

**Example:** ask “What is YOLO?” — the retrieved chunks and the question get woven into the prompt, `gpt-4o-mini` drafts an answer grounded in that context, and the parser hands back plain text — the provider changed, the shape of the pipeline didn't.

In [27]:
# Initialize an LLM instance using OpenAI's "gpt-4o-mini" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define a prompt template for generating answers based on a given context and question
prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """

# Create a ChatPromptTemplate from the string template, enabling dynamic input for context and question
prompt = ChatPromptTemplate.from_template(prompt_str)

# Create a retrieval pipeline to fetch relevant context and pass through the user's question
retrieval = RunnableParallel(
    {
        # Use the vector database as a retriever to fetch relevant context for the question
        "context": vectordb.as_retriever(),

        # Pass through the user's input question without modification
        "question": RunnablePassthrough()
    }
)

# Define an output parser to format the generated response into a string
output_parser = StrOutputParser()

# Create a processing chain that retrieves context, formats the prompt, generates an LLM response, and parses the output
chain = retrieval | prompt | llm | output_parser

### ▶️ Cell — Run the full RAG pipeline

Same as before — `.invoke(...)` runs the entire chain end to end and returns just the final text.

In [28]:
# Invoke the chain of components (retrieval, prompt generation, LLM processing, and output parsing)
# The question "What is YOLO?" is passed through the chain to generate the response
response = chain.invoke("What is YOLO?")

# Print the response generated by the chain
print(response)

YOLO, which stands for "You Only Look Once," is a unified model for real-time object detection. It frames object detection as a regression problem, predicting bounding boxes and class probabilities directly from full images in a single evaluation. YOLO is designed to be extremely fast, processing images at rates of 45 frames per second for the base model and over 150 frames per second for a smaller version called Fast YOLO. This speed allows it to operate in real-time with minimal latency, making it suitable for applications like video processing and interactive systems. YOLO also generalizes well to new domains, outperforming traditional detection methods when applied to different types of images, such as artwork compared to natural images. The entire model is trained jointly on a loss function that corresponds directly to detection performance, which contributes to its effectiveness and efficiency in object detection tasks.


## Other chain invoking methods!

.invoke(): The goal is to pass in an input and receive the output—neither more nor less.

.batch(): This is faster than using invoke three times when you wish to supply several inputs to get multiple outputs because it handles the parallelization for you.

.stream():  We may begin printing the response before the entire response is complete.

### 📚 Cell — Run the chain on multiple questions at once

Unchanged from the Cohere version — `.batch([...])` runs the same chain over several questions in one call, returning a list of answers in the same order.

In [29]:
response_with_batch = chain.batch(["What is Transformers", "How is Transformer different than YOLO?"])

for response in response_with_batch:
  print(response)
  print("\n")

Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language modeling and machine translation. Unlike traditional models that rely on recurrent neural networks (RNNs) or convolutional neural networks (CNNs), Transformers utilize a mechanism called self-attention to compute representations of input and output sequences. This architecture allows for significant parallelization, making it faster to train and capable of achieving state-of-the-art results in various tasks. The Transformer consists of an encoder-decoder structure, where the encoder processes the input sequence and the decoder generates the output sequence, both using stacked self-attention and fully connected layers. The model has demonstrated superior performance in translation tasks and generalizes well to other applications, such as constituency parsing.


Transformer and YOLO are both architectures used in the field of machine learning, but they serve different purpose

### 🌊 Cell — Stream the answer as it's generated

Also unchanged — `.stream(...)` yields pieces of the answer as `gpt-4o-mini` generates them, printed with `end=""` so the words appear one after another instead of all at once.

In [30]:
for chunk in chain.stream("What are the 3 vectors in Transformers architecture?"):
  print(chunk, flush=True, end="")

In the Transformer's architecture, the three vectors used in the attention mechanism are:

1. **Queries**: These are the vectors that the model uses to search for relevant information in the input data.
2. **Keys**: These vectors represent the input data and are compared against the queries to determine the relevance of each input position.
3. **Values**: These vectors contain the actual information that is retrieved based on the attention scores derived from the queries and keys.

In self-attention layers, all three vectors (queries, keys, and values) come from the same source, while in encoder-decoder attention layers, the queries come from the decoder and the keys and values come from the encoder.